# Agents as Tools

*Notebook 15*

Call specialists like functions — without giving up control of the conversation.

Agents-as-tools let an orchestrator call multiple specialists and combine their outputs into one response.
<br>
<br>
**Topics:**
- Handoff vs agent-as-tool — when to use each

- Calling agents like functions with `.as_tool()`

- Orchestrator pattern — one agent coordinates many specialists

- Combining multiple specialist results in one response

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

Handoffs work for routing — but once the specialist takes over, the orchestrator is done. Agents-as-tools let you call multiple specialists and combine their outputs without giving up control.

---

## 🔀 Part 1: Handoff vs Agent-as-Tool

Both patterns involve multiple agents. The difference is who stays in control.

**Handoff** — transfer of control:
```
User → Triage → Specialist (takes over, handles response)
```

**Agent as Tool** — orchestrator stays in control:
```
User → Orchestrator → calls Specialist A → gets result back
→ calls Specialist B → gets result back
→ synthesizes both results → responds to user
```

<div style="text-align: left; display: inline-block;">

| | Handoff | Agent as Tool |
|---|---|---|
| Who responds to user | Specialist | Orchestrator |
| Can combine specialist outputs | No | Yes |
| Orchestrator stays in control | No | Yes |
| Best for | Routing | Combining specialist outputs |

</div>

---

## 🛠️ Part 2: The `.as_tool()` Method

Any agent can be exposed as a tool using `.as_tool()`. The orchestrator calls it like any other tool — the specialist runs, returns its output, and the orchestrator uses that output to continue. The model phrases the input as a string, the specialist runs on that string as its message, then returns its output. In your own projects, the pattern is: build a specialist agent first, then expose it inside another agent's `tools=[...]` list with `specialist_agent.as_tool(...)` — and keep the specialist's instructions general enough to handle any input the orchestrator phrases.

In [ ]:
# Two specialist agents
pros_agent = Agent(
    name="ProsAnalyst",
    instructions="You identify and explain the advantages and benefits of any topic. Be specific and concise.",
    model=MODEL
)

cons_agent = Agent(
    name="ConsAnalyst",
    instructions="You identify and explain the disadvantages and risks of any topic. Be specific and concise.",
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Specialist agents defined")

#### Define the Orchestrator

In [ ]:
orchestrator_instructions = (
    "You provide balanced analysis on any topic.\n"
    "Always call both the pros_analyst and cons_analyst tools, then\n"
    "synthesize their outputs into a balanced summary with clear sections."
)

analysis_orchestrator = Agent(
    name="AnalysisOrchestrator",
    instructions=orchestrator_instructions,
    model=MODEL,
    tools=[
        pros_agent.as_tool(
            tool_name="pros_analyst",
            tool_description="Analyzes advantages and benefits of a topic."
        ),
        cons_agent.as_tool(
            tool_name="cons_analyst",
            tool_description="Analyzes disadvantages and risks of a topic."
        )
    ]
)

# The `tool_name` gives the orchestrator a stable, function-like name to call — independent of the specialist agent's display name, which keeps your orchestrator prompts clean.

# --------------------------------------------------------------
print("✅ Orchestrator ready")
print("   Tools: pros_analyst, cons_analyst")

#### Run the Orchestrator

In [ ]:

result = await Runner.run(analysis_orchestrator, input="Working from home vs working in an office")

print("🔀 ORCHESTRATOR RESULT")
print("=" * 60)
print(f"Handled by: {result.last_agent.name}")
print()
print(result.final_output)
print("=" * 60)

### 💡 Why This Works

The orchestrator called both specialists and synthesized their results. `result.last_agent.name` is the orchestrator in this pattern — not a specialist — because control never transferred away. This matters when you need a single final response that combines multiple specialist views — comparing pros and cons, gathering checks from different reviewers, or enforcing one consistent output format.

---

## 🏗️ Part 3: Multi-Specialist Orchestration

The real power of agents-as-tools is calling multiple specialists in sequence and building a richer output than any single agent could produce.

In [ ]:
# Three specialist agents for a product review system
features_agent = Agent(
    name="FeaturesAnalyst",
    instructions="You analyze product features. List and explain the key capabilities concisely.",
    model=MODEL
)

pricing_agent = Agent(
    name="PricingAnalyst",
    instructions="You analyze pricing and value. Comment on cost, tiers, and whether the pricing is fair.",
    model=MODEL
)

verdict_agent = Agent(
    name="VerdictWriter",
    instructions="You write final verdicts. Given a summary of features and pricing, write a clear buy/skip recommendation with reasoning.",
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Specialist agents defined")

#### Define Review Instructions

In [ ]:
review_instructions = (
    "You produce comprehensive product reviews.\n"
    "For any product:\n"
    "1. Call features_analyst to analyze the features\n"
    "2. Call pricing_analyst to evaluate the pricing\n"
    "3. Call verdict_writer to produce a final recommendation — it has the feature and pricing analyses available from the prior steps\n"
    "4. Present all three sections clearly to the user."
)

review_orchestrator = Agent(
    name="ReviewOrchestrator",
    instructions=review_instructions,
    model=MODEL,
    tools=[
        features_agent.as_tool(
            tool_name="features_analyst",
            tool_description="Analyzes product features and capabilities."
        ),
        pricing_agent.as_tool(
            tool_name="pricing_analyst",
            tool_description="Evaluates product pricing and value for money."
        ),
        verdict_agent.as_tool(
            tool_name="verdict_writer",
            tool_description="Writes a final buy/skip verdict given feature and pricing analysis."
        )
    ]
)

# --------------------------------------------------------------
print("✅ Review orchestrator ready")

#### Run the Review Pipeline

In [ ]:
product = "A wireless noise-cancelling headphone — 30-hour battery life, $249"

result = await Runner.run(review_orchestrator, input=product)

print("📋 PRODUCT REVIEW")
print("=" * 60)
print(f"Handled by: {result.last_agent.name}")
print()
print(result.final_output)
print("=" * 60)

### 💡 Why This Works

The orchestrator doesn't just call tools in parallel — within a single turn it holds the results of earlier tool calls in context, so it can summarize features and pricing before calling `verdict_writer`. The `verdict_writer` step is one option, not a requirement: the general pattern is gather specialist outputs first, then either synthesize them in the orchestrator or pass them to one final specialist.

---

## 💪 Practice Exercises

### Exercise 1: Trip Planner

*Covers: agents as tools — orchestrating specialist agents*

Build an orchestrator that combines input from an activities specialist and a food specialist to plan a day trip.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Trip Planner Orchestrator
# --------------------------------------------------------------
# Objective: Combine two specialists into one trip planning response.

# TODO 1: Create an ActivitiesExpert agent with instructions to suggest
#          sightseeing, outdoor, and cultural activities for a city

# TODO 2: Create a FoodExpert agent with instructions to suggest
#          restaurants, local dishes, and food experiences for a city

# TODO 3: Create a TripPlannerOrchestrator that calls both as tools
#          and combines their output into a one-day itinerary

# TODO 4: Run with: "Plan a day trip to New Orleans"
#          Print result.last_agent.name and result.final_output

# --- Write your code below this line ---

### Exercise 2: Job Application Reviewer

*Covers: agents as tools — multi-specialist orchestration*

Build an orchestrator that evaluates a resume using two specialist reviewers and produces a hiring recommendation.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Job Application Reviewer
# --------------------------------------------------------------
# Objective: Coordinate two reviewers into one hiring recommendation.

resume = """
Candidate: Alex Chen
Experience: 3 years Python development, 1 year team lead
Skills: Python, FastAPI, PostgreSQL, Docker, AWS basics
Education: BS Computer Science
Projects: Built internal reporting tool used by 50+ employees
"""

job_requirements = """
Role: Senior Python Developer
Required: 4+ years Python, REST API experience, database skills
Nice to have: Cloud experience, leadership experience
"""

# TODO 1: Create a SkillsReviewer agent that checks technical fit

# TODO 2: Create an ExperienceReviewer agent that checks seniority fit

# TODO 3: Create a HiringOrchestrator that calls both and produces
#          a final hire/pass recommendation with reasoning

# TODO 4: Run with the resume and job requirements above

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Two patterns, different control:**

- Handoffs transfer control — the specialist owns the conversation

- Agents as tools keep the orchestrator in control — specialists are callable functions

- Use handoffs when one specialist should own the conversation; use agents-as-tools when the orchestrator needs to gather and combine multiple specialist outputs
<br>
<br>
**as_tool() syntax:**

- `agent.as_tool(tool_name="...", tool_description="...")` exposes any agent as a tool

- The `tool_description` tells the orchestrator when and why to call it
<br>
<br>
**Orchestrator pattern:**

- One orchestrator, multiple specialists — each focused on one task

- The orchestrator synthesizes results into a single coherent response

- `result.last_agent.name` is the orchestrator in this workflow

---

## 📍 Next Step

**Notebook 16: Parallel Execution**  

Run multiple agents concurrently with asyncio and combine their results for faster, more powerful workflows.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-15-agents-as-tools)

---